<a href="https://colab.research.google.com/github/paridhietal/TransferLearning/blob/main/transferLearning2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers --quiet

In [ ]:
import chromadb
import json
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload memory_export.json

# now load it into a variable
with open("memory_export.json", "r") as f:
    memory = json.load(f)

print(f"Loaded {len(memory)} tasks from Notebook 1")

Saving memory_export.json to memory_export.json
Loaded 5 tasks from Notebook 1


In [ ]:
texts = [m["text"] for m in memory]
print(texts)

['Write a Python function that reverses a string', 'Write a Python function that checks if a string is a palindrome', 'Explain gradient descent in simple terms for a beginner', 'Write a Python function to count vowels in a string', 'Write a Python function that finds the largest number in a list']


In [ ]:
# load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# convert texts to embeddings
embeddings = model.encode(texts)
print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(5, 384)


In [ ]:
import os

# Define a path for persistence
CHROMA_DB_PATH = "./chroma_db_data"

# Create the directory if it doesn't exist
os.makedirs(CHROMA_DB_PATH, exist_ok=True)

# Initialize a persistent client
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Delete the collection if it already exists to ensure fresh data with correct schema
try:
    client.delete_collection(name="running")
    print("Existing 'running' collection deleted.")
except Exception as e:
    print(f"Could not delete collection (it might not exist yet): {e}")

# Get or create the collection (it will be created if deleted above)
collection = client.get_or_create_collection(name="running")

collection.add(
    ids        = [m["id"] for m in memory],
    embeddings = embeddings.tolist(),
    documents  = texts,
    metadatas  = [
        {
            "score": m["score"],
            "lesson": m["lesson"],
            "task": m["text"],   # Add 'task' from m["text"]
            "output": m["output"] # Add 'output' from m["output"]
        }
        for m in memory
    ]
)

print(f"Stored {len(memory)} tasks in chromaDB")

Could not delete collection (it might not exist yet): Collection [running] does not exist
Stored 5 tasks in chromaDB


In [ ]:
#Retrieval function
def retrieve_relevant_memory(query, top_k=3):
  results = collection.query(
      query_texts=[query],
      n_results=top_k
  )

  hits = []
  for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    hits.append({
        "task": meta["task"],
        "output": meta["output"],
        "lesson": meta["lesson"],
        "score": meta["score"],
        "distance": results["distances"][0][i]
    })
  return hits

#test it
hits = retrieve_relevant_memory("RAG")
for h in hits:
  print(f"[score={h['score']}] {h['task']}\n lesson: {h['lesson']}\n")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 24.6MiB/s]


[score=0.9] Write a Python function that checks if a string is a palindrome
 lesson: Handle edge cases: ignore spaces, punctuation, and capitalization.

[score=0.85] Write a Python function that reverses a string
 lesson: Good clean solution. Include a docstring and example usage next time.

[score=0.88] Write a Python function to count vowels in a string
 lesson: Correct solution. Add type hints and handle uppercase vowels explicitly.



In [ ]:
#Add new entries to ChromaDB after each run
def update_memory(new_id, task, output, score, lesson):
  collection.add(
      ids=[str(new_id)],
      documents=[f"{task} {lesson}"],
      metadatas=[{
          "task": task,
          "output": output,
          "lesson": lesson,
          "score": str(score)
      }]
  )
  print(f"Memory updated. Total: {collection.count()} entries")

In [ ]:
#Persist so it survives the session
# For PersistentClient, changes are automatically saved. No explicit .persist() call is needed.
# However, it's good practice to ensure the client is properly shut down if needed, but ChromaDB handles flush automatically.
print("ChromaDB data is being persisted to ./chroma_db_data")

ChromaDB data is being persisted to ./chroma_db_data
